# 1 Emulador de stream de datos

In [1]:
!python --version

Python 3.8.20


In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder\
    .master("spark://spark-master:7077")\
    .appName("Stream")\
    .config("spark.jars", "/opt/spark/jars/mysql-connector-j-8.4.0.jar")\
    .getOrCreate()  

/usr/local/lib/python3.8/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
26/08/07 20:32:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
import pyspark
import os

print("PySpark:", pyspark.__version__)
print("Spark:", spark.version)

PySpark: 3.5.1
Spark: 3.5.1


In [9]:
ultimo_id = 68883
consulta = f"""(
SELECT *
FROM orders
WHERE order_id > {ultimo_id}
) t
"""
df_orders = (
    spark.read
         .format("jdbc")
         .option("url", "jdbc:mysql://mysql:3306/retail_db")
         .option("dbtable", consulta)
         .option("user", "root")
         .option("password", "root")
         .option("driver", "com.mysql.cj.jdbc.Driver")
         .load()
)

df_orders.show()

[Stage 0:>                                                          (0 + 1) / 1]

+--------+----------+-----------------+------------+
|order_id|order_date|order_customer_id|order_status|
+--------+----------+-----------------+------------+
+--------+----------+-----------------+------------+



## Correr emulador de datos en una terminal
    python emulador_datos.py

# 2 Kafka creacion de un PRODUCTOR aleatorio

- Kafka es un sistema de mensajería distribuido que recibe eventos de los productores
  
- Los almacena temporalmente en tópicos

  
- Permite que uno o varios consumidores los procesen de forma independiente y en tiempo real.

## A Creacion de un topico 

#### 1. ENtrar al contenedor de kafka 
docker exec -it kafka bash

#### 2. Enlistar topicos 
kafka-topics --bootstrap-server localhost:9092 --list

#### 3. Crea topico ventas 

kafka-topics --create --topic ventas --bootstrap-server localhost:9092 --partitions 2 --replication-factor 1

#### 4. Visualizdor en kafka como llegan los datos 

kafka-console-consumer --bootstrap-server kafka:9092 --topic ventas --from-beginning


## B En el notebook

In [1]:
!pip install kafka-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 614.2/614.2 kB 3.3 MB/s eta 0:00:0000:01

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [9]:
from kafka import KafkaProducer
import json
import time
import random
from datetime import datetime
#Crea conexión con Kafka
#Se conecta al broker Kafka:

producer = KafkaProducer(
    bootstrap_servers="kafka:9092",
    value_serializer=lambda x: json.dumps(x).encode("utf-8")
)

while True:
    # Genera una venta ficticia cada 3 segundos
    venta = {
        "order_id": random.randint(10000,99999),
        "customer_id": random.randint(1,1000),
        "monto": round(random.uniform(10,500),2),
        "fecha": str(datetime.now())
    }
    
    # Envía el mensaje a Kafka
    producer.send(
        "ventas",
        value=venta
    )


    print("Enviado:", venta)


    time.sleep(8)

Enviado: {'order_id': 47682, 'customer_id': 416, 'monto': 237.28, 'fecha': '2026-08-05 01:39:57.470583'}
Enviado: {'order_id': 52568, 'customer_id': 312, 'monto': 87.3, 'fecha': '2026-08-05 01:40:05.581517'}
Enviado: {'order_id': 48692, 'customer_id': 905, 'monto': 426.46, 'fecha': '2026-08-05 01:40:13.584388'}
Enviado: {'order_id': 99135, 'customer_id': 295, 'monto': 453.78, 'fecha': '2026-08-05 01:40:21.593550'}
Enviado: {'order_id': 75940, 'customer_id': 123, 'monto': 470.3, 'fecha': '2026-08-05 01:40:29.601680'}
Enviado: {'order_id': 68412, 'customer_id': 556, 'monto': 38.89, 'fecha': '2026-08-05 01:40:37.607686'}
Enviado: {'order_id': 57212, 'customer_id': 253, 'monto': 475.76, 'fecha': '2026-08-05 01:40:45.615709'}
Enviado: {'order_id': 24882, 'customer_id': 399, 'monto': 271.09, 'fecha': '2026-08-05 01:40:53.623416'}
Enviado: {'order_id': 78251, 'customer_id': 373, 'monto': 288.09, 'fecha': '2026-08-05 01:41:01.629117'}


KeyboardInterrupt: 

# 3 Kafka guarda el evento en el tópico: 

In [2]:
!pip install kafka-python mysql-connector-python


[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


#### 1. ENtrar al contenedor de kafka 
    docker exec -it kafka bash

#### 2. Enlistar topicos 
    kafka-topics --bootstrap-server localhost:9092 --list

#### 3. Crea topico order , orders_items

    kafka-topics --create --topic orders_topic --bootstrap-server kafka:9092 --partitions 2 --replication-factor 1
    
    kafka-topics --create --topic order_items_topic --bootstrap-server kafka:9092 --partitions 2 --replication-factor 1
    
#### 4. Ejecutar  mysql_kafka_producer.py

    

#### 5. Visualizdor en kafka como llegan los datos 

    kafka-console-consumer --bootstrap-server kafka:9092 --topic orders_topic --from-beginning

## PRACTICA 

### 1 Crear un emulador de datos para otra estructura de la base de datos retail_db
### 2 Crear un topico en kafka para mostrar datos de dicha estructura
### 3 Ver los datos insertados en mysql por el topico 